# Energy-gap law para los $k_{NR}$ del `ErIon`

1. Tomo los $k_{NR}$ (a $T_0$) de cada `TemperatureDependentInternalRelaxationRef` del modelo y los grafico contra el gap $\Delta E$.
2. Ajusto $k_{NR}(T_0) = B\, e^{-a \Delta E}$ (lineal en $\ln k$).
3. Heatmap de $e^{-a\Delta E}\cdot F(T)$ para todo par de niveles, con el factor de temperatura en la misma convención `Ref` que usa el modelo
   (el ajuste ya está hecho a $T_0$, así que el factor se normaliza a $T_0$ para no contar dos veces la ocupación fonónica):

$$
F_{\downarrow}(T) = \left[\frac{1 + n(T)}{1 + n(T_0)}\right]^{p}, \qquad
F_{\uparrow}(T) = \left[\frac{n(T)}{1 + n(T_0)}\right]^{p}, \qquad
n(T) = \frac{1}{e^{\hbar\omega/k_BT} - 1}, \quad p = \frac{\Delta E}{\hbar\omega}
$$

$F_\uparrow$ es $F_\downarrow$ por balance detallado ($[n/(1+n)]^p = e^{-\Delta E/k_BT}$). A $T=T_0$, $F_\downarrow = 1$.

In [ ]:
from upconversion_models.ions.erbium import ErIon
from upconversion_models.transitions import TemperatureDependentInternalRelaxationRef
from upconversion_models.utils import h, c, k, hc
from jablonski import SingletState
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import numpy as np
import pint

u = pint.get_application_registry()
er = ErIon()

## 1. $k_{NR}$ del modelo vs $\Delta E$

In [ ]:
names, gaps, knr = [], [], []
for t in er._yield(TemperatureDependentInternalRelaxationRef):
    names.append(f"{t.source}→{t.target}")
    gaps.append(t._gap_wavenumber.default.to(1 / u.cm).magnitude)
    knr.append(t.reference_rate.default.to(1 / u.s).magnitude)

names, gaps, knr = np.array(names), np.array(gaps), np.array(knr)
phonon_wavenumber = next(er._yield(TemperatureDependentInternalRelaxationRef)).phonon_wavenumber.default.to(1 / u.cm).magnitude
T0 = ErIon.T0.default.to(u.K).magnitude

for n_, g, kk in zip(names, gaps, knr):
    print(f"{n_:>10}  ΔE = {g:6.0f} cm⁻¹   k_nr = {kk:.3g} s⁻¹")
print(f"\nħω = {phonon_wavenumber} cm⁻¹, T0 = {T0} K")

## 2. Ajuste exponencial

Ajuste lineal de $\ln k$ vs $\Delta E$. Las transiciones con $k_{NR}=0$ (Er5→Er4) no entran.
`EXCLUDE` permite sacar outliers del ajuste que se usa en el heatmap (p.ej. `{"Er8→Er7"}`); la curva punteada muestra siempre el ajuste con todos los puntos para comparar.

In [ ]:
EXCLUDE = set()  # e.g. {"Er8→Er7"}


def fit(mask):
    slope, intercept = np.polyfit(gaps[mask], np.log(knr[mask]), 1)
    return -slope, np.exp(intercept)  # a [cm], B [1/s]


valid = knr > 0
a_all, B_all = fit(valid)
a, B = fit(valid & ~np.isin(names, list(EXCLUDE)))

print(f"todos:       a = {a_all:.3e} cm,  B = {B_all:.3e} s⁻¹")
print(f"sin EXCLUDE: a = {a:.3e} cm,  B = {B:.3e} s⁻¹")

dE = np.linspace(0, 5000, 200)
fig, ax = plt.subplots()
ax.scatter(gaps[valid], knr[valid], zorder=3)
for n_, g, kk in zip(names[valid], gaps[valid], knr[valid]):
    ax.annotate(n_, (g, kk), textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.plot(dE, B * np.exp(-a * dE), label=rf"$B e^{{-a\Delta E}}$, a = {a:.2e} cm, B = {B:.2e} s$^{{-1}}$")
if EXCLUDE:
    ax.plot(dE, B_all * np.exp(-a_all * dE), ls="--", c="gray", label="todos los puntos")
ax.set_yscale("log")
ax.set_xlabel(r"$\Delta E$ (cm$^{-1}$)")
ax.set_ylabel(r"$k_{NR}(T_0)$ (s$^{-1}$)")
ax.legend(fontsize=8)
plt.show()

residuals = np.log10(knr[valid]) - np.log10(B * np.exp(-a * gaps[valid]))
for n_, r in zip(names[valid], residuals):
    print(f"{n_:>10}  log10(k / fit) = {r:+.2f}")

## 3. Heatmap $e^{-a\Delta E}\cdot F(T)$

Fila = nivel de origen, columna = destino (misma convención que `temperature_dependance.ipynb`). Debajo de la diagonal: relajación (emisión de fonones), arriba: excitación térmica.
Las celdas con borde rojo son las transiciones no radiativas que el modelo ya incluye. Multiplicando por $B$ se obtiene la tasa estimada en s⁻¹.

In [ ]:
levels = [l for l in er._yield(SingletState)][:10]
E = np.array([(l.energy / hc).to(1 / u.cm).magnitude for l in levels])
kB = (k / hc).to(1 / (u.cm * u.K)).magnitude  # cm⁻¹ / K

existing = {(str(t.source), str(t.target)) for t in er._yield(TemperatureDependentInternalRelaxationRef)}


def occupation(T):
    return 1 / np.expm1(phonon_wavenumber / (kB * T))


def gap_law_heatmap(T, ceil_p=False):
    n, n0 = occupation(T), occupation(T0)
    V = np.full((len(levels), len(levels)), np.nan)
    for i, Ei in enumerate(E):
        for j, Ej in enumerate(E):
            if i == j:
                continue
            dE = abs(Ei - Ej)
            p = dE / phonon_wavenumber
            if ceil_p:
                p = np.ceil(p)
            F = ((1 + n) / (1 + n0)) ** p if Ei > Ej else (n / (1 + n0)) ** p
            V[i, j] = np.exp(-a * dE) * F
    return V


def plot_heatmap(T, vmin=1e-30, **kwargs):
    V = gap_law_heatmap(T, **kwargs)
    labels = [repr(l) for l in levels]
    fig, ax = plt.subplots(figsize=(9, 7.5))
    norm = LogNorm(vmin=vmin, vmax=1)
    im = ax.imshow(V, norm=norm)
    for i in range(V.shape[0]):
        for j in range(V.shape[1]):
            if np.isfinite(V[i, j]):
                ax.text(j, i, f"{np.log10(V[i, j]):.1f}", ha="center", va="center", fontsize=8,
                        color="k" if norm(V[i, j]) > 0.6 else "w")
            if (labels[i], labels[j]) in existing:
                ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="r", lw=2))
    fig.colorbar(im, label=r"$e^{-a\Delta E}\,F(T)$", extend="min")
    ax.set_xticks(np.arange(len(levels)), labels)
    ax.set_yticks(np.arange(len(levels)), labels)
    ax.set_ylabel("from")
    ax.set_xlabel("to")
    ax.set_title(
        rf"$\log_{{10}}[e^{{-a\Delta E}} F(T)]$, T = {T} K, "
        rf"$\hbar\omega$ = {phonon_wavenumber:g} cm$^{{-1}}$, a = {a:.2e} cm"
    )
    plt.show()
    return V


V = plot_heatmap(T0)

Mismo heatmap a otra temperatura (para ver qué transiciones se prenden al calentar):

In [ ]:
V_hot = plot_heatmap(400)